# <center> SIGNED LD FOR COLOCALIZATION AND FINE MAPPING <center>

Builds a **signed** LD matrix for the C5orf67 locus from 1000 Genomes EUR genotypes, and
harmonises four GWAS onto it. The output is a matched set of inputs for SharePro and CARMA:
one correlation matrix, one variant order, and four sets of betas that all refer to the same
allele.

Footprint on a laptop: about 2 MB downloaded, a few seconds of compute, peak memory a
251 x 251 matrix.

## Why the sign matters

Both SharePro and CARMA model association statistics as a function of the *signed* correlation
between variants. The sign says whether the effects of two correlated variants reinforce or
cancel. Without it, two variants in near-perfect linkage whose Z scores point in opposite
directions cannot be reconciled, and a fine mapper is forced to read each as its own signal.

No LDlink endpoint returns a signed matrix: `ldmatrix` gives R2 or D', both unsigned, and
recovering the sign from `ldpair` or `ldproxy` means reconstructing one phase per variant and
assuming `sign(r_ij) = s_i s_j`, which holds exactly only inside a tight linkage block.

`plink --r square` computes the signed correlation for every pair directly from the
haplotypes. No reconstruction, no assumption, and every pair is exact rather than just the
strongly linked ones. The cost is a small download, which tabix keeps to the region rather
than the chromosome.

## The one thing to get right

Every sign in the matrix refers to PLINK's **A1** allele. A trait's beta refers to its own
effect allele. Where the two differ the beta must be flipped, otherwise a correctly signed
matrix is no better than an unsigned one. Section 6 does that, checks both alleles so a
genuine allele mismatch is caught rather than silently strand-flipped, and reports the count.

## What the notebook does

| Section | Step |
|---|---|
| 1-2 | Configure paths and the region, install PLINK and htslib |
| 3 | Download and slice the four GWAS to the region |
| 4 | Pull the 1000 Genomes region and build the EUR sample list |
| 5 | PLINK: genotypes to a signed correlation matrix |
| 6 | Flip every beta onto PLINK's A1 |
| 7 | Test the matrix against the summary statistics |
| 8-9 | Write the inputs, delete the scratch |

## Scope

The install and download cells were written against the documented interfaces of PLINK,
htslib and the 1000 Genomes FTP, and were not executed here. Everything downstream of
`region_eur.bim` was tested, including on simulated genotypes with randomised allele
labelling.


---
## 1. Configuration

In [ ]:
# @title Paths, region and options
import os, io, sys, platform, shutil, subprocess, urllib.request
from pathlib import Path
import numpy as np, pandas as pd

# ---- project layout ----------------------------------------------------------
# 2_colocalization_and_finemapping/
#   dat/                        <- ALL inputs, read here and written back here
#       raw/                    <- region slices of the source sumstats
#   2a_colocalization/          <- this notebook lives here
#       interim/                <- PLINK scratch, safe to delete
#           bin/                <- downloaded plink binary
#       results/                <- SharePro output and the diagnostic plot
#   2b_finemapping/             <- CARMA_T2D_signed.R, with its own interim/ and results/
#
# The prepared inputs (signed LD, harmonised betas, variant_order.tsv) go back into dat/
# rather than into interim/, because both 2a and 2b read them. interim/ is scratch only.
HERE    = Path.cwd()
ROOT    = HERE.parent                    # 2_colocalization_and_finemapping
DAT     = ROOT / 'dat'
WORK    = HERE / 'interim'; WORK.mkdir(exist_ok=True)
RESULTS = HERE / 'results'; RESULTS.mkdir(exist_ok=True)
BIN     = WORK / 'bin';     BIN.mkdir(exist_ok=True)

assert DAT.is_dir(), (f'expected the data folder at {DAT}. Set DAT by hand if your layout '
                      'differs.')

# ---- the target variants -----------------------------------------------------
# One rsID per line under a SNP header: the LD proxies of rs3843467 that define this locus.
# This is the only thing the notebook needs fixed in advance. Every beta, allele and frequency
# is re-derived from the sources in section 3, so no effect estimate enters from here.
VARIANT_LIST = DAT / 'variant_list.txt'

# ---- summary statistics ------------------------------------------------------
# Harmonisation needs to know which allele each BETA refers to, so section 3 downloads the
# source summary statistics and slices out the region.
RAW_DIR = DAT / 'raw'; RAW_DIR.mkdir(exist_ok=True)

# The three MAGIC traits are on the GWAS Catalog FTP and are fetched automatically.
# key: (GCST, EFO, PMID, N)
CATALOG = {
    'fasting_insulin': ('GCST90002238', 'EFO_0004467', '34059833', 151013),
    'fasting_glucose': ('GCST90002232', 'EFO_0004468', '34059833', 200622),
    '2h_glucose':      ('GCST90002227', 'EFO_0004307', '34059833',  63396),
}
CATALOG_COLS = dict(id_col='variant_id', chrom='chromosome', pos='base_pair_location',
                    ea='effect_allele', oa='other_allele', eaf='effect_allele_frequency',
                    beta='beta', se='standard_error')

# DIAMANTE sits behind an access form at https://diagram-consortium.org/downloads.html, so it
# cannot be fetched with wget. Section 3 slices the region out of the copy you downloaded and
# leaves the original alone.
#
# Header, space delimited, alleles in lower case:
#   chromosome(b37) position(b37) chrposID rsID effect_allele other_allele
#   effect_allele_frequency Fixed-effects_beta Fixed-effects_SE Fixed-effects_p-value
DIAMANTE_FILE = DAT / 't2d_gwas_DIAMANTE-EUR.sumstat.txt'
DIAMANTE_N    = 177201
DIAMANTE_COLS = dict(id_col='rsID', chrom='chromosome(b37)', pos='position(b37)',
                     ea='effect_allele', oa='other_allele', eaf='effect_allele_frequency',
                     beta='Fixed-effects_beta', se='Fixed-effects_SE')

# Filled in by section 3. Each value is (sliced_path, columns_dict, N).
RAW_SUMSTATS = {}

# ---- region ------------------------------------------------------------------
# GRCh37, the build of both the phase 3 VCFs and your summary statistics. rs3843467 sits at
# chr5:55,856,375. The original proxy list was +/- 72 kb, so 120 kb leaves room to spare.
CHROM, CENTRE, HALF_WIDTH = 5, 55_856_375, 120_000
START, END = CENTRE - HALF_WIDTH, CENTRE + HALF_WIDTH

SUPERPOP = 'EUR'
AMBIGUOUS_MAF_MARGIN = 0.05   # A/T and C/G need |freq - 0.5| above this to be resolvable

print(f'region   chr{CHROM}:{START:,}-{END:,}  (GRCh37)')
print(f'dat      {DAT.resolve()}')
print(f'interim  {WORK.resolve()}')
print(f'results  {RESULTS.resolve()}\n')

assert VARIANT_LIST.exists(), (
    f'expected the target variant list at {VARIANT_LIST}. It is one rsID per line under a '
    'SNP header.')
print(f'variants  {VARIANT_LIST.name}, '
      f'{sum(1 for _ in open(VARIANT_LIST)) - 1} rsIDs')
print('summary statistics are fetched in section 3')


---
## 2. Install PLINK and htslib

PLINK 1.9 rather than 2, because 2 renamed the correlation flags and `--r square` is what gives
a signed matrix. Conda is tried first since it works the same on Linux and macOS, then a direct
binary download as a fallback.

Equivalent shell, if you would rather run it outside the notebook:

```bash
# conda, either platform
conda install -y -c bioconda plink htslib bcftools

# or Debian and Ubuntu
sudo apt-get update && sudo apt-get install -y plink1.9 tabix bcftools

# or the binary straight from the PLINK site
wget https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20231211.zip
unzip -o plink_linux_x86_64_20231211.zip plink && chmod +x plink
```


In [ ]:
# @title Install
def have(tool):
    return shutil.which(tool) or (BIN / tool).exists()

PLINK_RETURN = {   # plink_common.h, so a bare exit code is readable
    1: 'out of memory', 2: 'could not open a file', 3: 'invalid file format',
    4: 'calculation not supported', 5: 'invalid command line', 6: 'write failure',
    7: 'read failure', 9: 'thread creation failed', 10: 'allele mismatch',
    11: 'nothing to calculate', 12: 'ALL SAMPLES EXCLUDED (--keep matched nothing)',
    13: 'ALL VARIANTS EXCLUDED (--extract matched nothing)', 14: 'network error',
}

def run(cmd, check=True, quiet=False):
    """Run a shell command and show its output in the notebook, not the terminal."""
    print('$', cmd)
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (proc.stdout or '') + (proc.stderr or '')
    if out.strip() and not quiet:
        print('\n'.join('  ' + l for l in out.strip().splitlines()[-30:]))
    if proc.returncode and check:
        meaning = PLINK_RETURN.get(proc.returncode, '')
        raise SystemExit(f'\ncommand failed with exit status {proc.returncode}'
                         + (f'  [{meaning}]' if meaning else ''))
    return proc

os.environ['PATH'] = f"{BIN.resolve()}{os.pathsep}{os.environ['PATH']}"

# --- htslib, for tabix and bgzip ---------------------------------------------
if not (have('tabix') and have('bgzip')):
    if shutil.which('conda'):
        run('conda install -y -q -c bioconda htslib bcftools', check=False)
    elif shutil.which('apt-get'):
        run('apt-get -qq update && apt-get -qq install -y tabix bcftools', check=False)
    elif shutil.which('brew'):
        run('brew install htslib bcftools', check=False)

# --- PLINK 1.9 ----------------------------------------------------------------
if not have('plink'):
    if shutil.which('conda'):
        run('conda install -y -q -c bioconda plink', check=False)

if not have('plink'):
    system, machine = platform.system(), platform.machine()
    asset = ('plink_mac_arm64_20231211.zip' if system == 'Darwin' and machine == 'arm64' else
             'plink_mac_20231211.zip'       if system == 'Darwin' else
             'plink_linux_x86_64_20231211.zip')
    url = f'https://s3.amazonaws.com/plink1-assets/{asset}'
    print(f'downloading {url}')
    run(f'wget -q "{url}" -O {WORK}/plink.zip && unzip -o -q {WORK}/plink.zip plink '
        f'-d {BIN} && chmod +x {BIN}/plink', check=False)

PLINK = str(BIN / 'plink') if (BIN / 'plink').exists() else (
        shutil.which('plink') or shutil.which('plink1.9') or '')
TABIX = shutil.which('tabix') or str(BIN / 'tabix')

for label, path in (('plink', PLINK), ('tabix', TABIX)):
    print(f'  {label:6s} {path if path else "NOT FOUND"}')
if not PLINK:
    raise SystemExit('PLINK not found. Install it with one of the shell commands above.')


---
## 3. Summary statistics, downloaded and sliced

The three MAGIC traits come off the GWAS Catalog FTP. The exact filename is discovered from the
directory index rather than guessed, because the Catalog is inconsistent about whether it writes
`build37` or `Build37`.

If a tabix index is published alongside the file, only the bytes for the region are pulled, a
few kilobytes. If not, the file is streamed and filtered on the fly, so the multi-hundred-megabyte
original is never written to disk.

DIAMANTE cannot be automated, since it sits behind an access form at
<https://diagram-consortium.org/downloads.html>. It is read from the copy already in `dat/`,
named by `DIAMANTE_FILE` in section 1, and the original is left alone. That file is space
delimited with lower case alleles, both of which are handled.

Everything written here is a region slice of a few hundred kilobytes. Section 9 deletes the
rest.


In [ ]:
# @title Discover and slice the GWAS Catalog files
import gzip, re, csv

GWAS_FTP = 'https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics'

def catalog_dir(gcst):
    """GCST90002238 -> GCST90002001-GCST90003000, the Catalog's thousand-wide buckets."""
    digits = gcst.replace('GCST', '')
    width = len(digits)                 # the Catalog keeps the accession's own zero padding
    num = int(digits)
    lo = ((num - 1) // 1000) * 1000 + 1
    return f'GCST{lo:0{width}d}-GCST{lo + 999:0{width}d}'

def find_build37(gcst, efo, pmid):
    """Read the harmonised directory index and return (url, tbi_url or None)."""
    base = f'{GWAS_FTP}/{catalog_dir(gcst)}/{gcst}/harmonised/'
    try:
        with urllib.request.urlopen(base, timeout=120) as fh:
            html = fh.read().decode('utf8', 'replace')
    except Exception as exc:
        raise SystemExit(f'could not list {base}: {exc}')
    names = set(re.findall(r'href="([^"]+)"', html))
    names |= set(re.findall(r'([\w.\-]+\.tsv\.gz(?:\.tbi)?)', html))
    wanted = re.compile(rf'{pmid}-{gcst}-{efo}-[Bb]uild37\.f\.tsv\.gz$')
    hit = next((n for n in names if wanted.search(n)), None)
    if hit is None:   # fall back to any build37 file for this study
        hit = next((n for n in names
                    if re.search(r'-[Bb]uild37\.f\.tsv\.gz$', n) and gcst in n), None)
    if hit is None:
        raise SystemExit(f'no build37 file found in {base}. Files seen: '
                         f'{sorted(n for n in names if n.endswith("tsv.gz"))[:6]}')
    hit = hit.split('/')[-1]
    tbi = hit + '.tbi' if any(n.split('/')[-1] == hit + '.tbi' for n in names) else None
    return base + hit, (base + tbi if tbi else None)

def slice_stream(url, out_path, cols):
    """Stream the gzip and keep only the rows in the region. Constant memory, nothing kept."""
    keep = 0
    with urllib.request.urlopen(url, timeout=600) as raw, \
         gzip.open(raw, 'rt') as fh, open(out_path, 'w', newline='') as out:
        reader = csv.reader(fh, delimiter='\t')
        header = next(reader)
        writer = csv.writer(out, delimiter='\t')
        writer.writerow(header)
        ci, pi = header.index(cols['chrom']), header.index(cols['pos'])
        for row in reader:
            if not row or row[ci] != str(CHROM):
                continue
            try:
                p = int(float(row[pi]))
            except ValueError:
                continue
            if START <= p <= END:
                writer.writerow(row); keep += 1
    return keep

for name, (gcst, efo, pmid, N) in CATALOG.items():
    out = RAW_DIR / f'{name}_chr{CHROM}_{START}_{END}.tsv'
    if out.exists() and out.stat().st_size > 200:
        n_rows = sum(1 for _ in open(out)) - 1
        print(f'{name:17s} already sliced, {n_rows} rows')
        RAW_SUMSTATS[name] = (out, CATALOG_COLS, N); continue

    url, tbi = find_build37(gcst, efo, pmid)
    print(f'{name:17s} {url.split("/")[-1]}')
    if tbi:
        print('                  tabix index found, pulling the region only')
        header = subprocess.run(f'{TABIX} -H "{url}"', shell=True, capture_output=True,
                                text=True).stdout
        body = subprocess.run(f'{TABIX} "{url}" {CHROM}:{START}-{END}', shell=True,
                              capture_output=True, text=True).stdout
        if body.strip():
            out.write_text((header if header.strip() else '') + body)
        else:
            print('                  tabix returned nothing, falling back to streaming')
            tbi = None
    if not tbi:
        print('                  streaming and filtering, the full file is never written')
        slice_stream(url, out, CATALOG_COLS)
    n_rows = sum(1 for _ in open(out)) - 1
    print(f'                  {n_rows} rows in region, {out.stat().st_size/1024:.0f} KB')
    RAW_SUMSTATS[name] = (out, CATALOG_COLS, N)


In [ ]:
# @title Slice DIAMANTE to the region
diam_out = RAW_DIR / f'T2D_DIAMANTE_chr{CHROM}_{START}_{END}.tsv'

if diam_out.exists() and diam_out.stat().st_size > 200:
    print(f'already sliced, {sum(1 for _ in open(diam_out)) - 1} rows')
else:
    src = Path(DIAMANTE_FILE) if Path(DIAMANTE_FILE).exists() else next(
        (p for place in (DAT, RAW_DIR, HERE, ROOT, Path.home() / 'Downloads')
         for pat in ('*DIAMANTE-EUR*sumstat*', '*DIAMANTE_EUR*sumstat*')
         for p in sorted(place.glob(pat))
         if p.is_file() and p.suffix.lower() in ('.txt', '.tsv', '.gz')), None)
    if src is None:
        raise SystemExit(
            f'DIAMANTE not found at {DIAMANTE_FILE}.\n'
            '  It is behind an access form, so it cannot be downloaded automatically.\n'
            '  Use the European form at https://diagram-consortium.org/downloads.html,\n'
            f'  then set DIAMANTE_FILE or drop the file in {DAT}.')

    print(f'source {src.name}  ({src.stat().st_size/1e6:.0f} MB)')
    opener = gzip.open if src.suffix == '.gz' else open

    with opener(src, 'rt') as fh:
        first = fh.readline().rstrip('\n')
    sep = '\t' if first.count('\t') > first.count(' ') else None   # None splits on whitespace
    header = first.split('\t') if sep else first.split()
    print(f'  delimiter: {"tab" if sep else "whitespace"}, {len(header)} columns')

    for role, col in DIAMANTE_COLS.items():
        if col not in header:
            raise SystemExit(f'column {col!r} (needed as {role}) is not in the header:\n'
                             f'  {header}')
    ci, pi = header.index(DIAMANTE_COLS['chrom']), header.index(DIAMANTE_COLS['pos'])
    widest = max(header.index(c) for c in DIAMANTE_COLS.values())

    # Genome wide and large, so it is streamed rather than read in. DIAMANTE is sorted by
    # chromosome then position, which lets us stop once past the region. Sortedness is checked
    # as we go, and the early exit is dropped if it ever fails.
    kept = seen = 0
    entered, sorted_so_far, prev_key = False, True, (-1, -1)
    with opener(src, 'rt') as fh, open(diam_out, 'w') as out:
        fh.readline()
        out.write('\t'.join(header) + '\n')
        for line in fh:
            row = line.split(sep) if sep else line.split()
            seen += 1
            if seen % 5_000_000 == 0:
                print(f'  {seen/1e6:.0f}M rows scanned, {kept} kept')
            if len(row) <= widest:
                continue
            chrom_txt = row[ci]
            try:
                c, p = int(chrom_txt), int(float(row[pi]))
            except ValueError:
                sorted_so_far = False
                continue
            if sorted_so_far and (c, p) < prev_key:
                sorted_so_far = False
            prev_key = (c, p)
            if chrom_txt == str(CHROM) and START <= p <= END:
                out.write('\t'.join(x.strip() for x in row) + '\n')
                kept += 1
                entered = True
            elif entered and sorted_so_far and (c > CHROM or (c == CHROM and p > END)):
                print('  past the region on a sorted file, stopping early')
                break
    print(f'  {seen:,} rows scanned, {kept} in region, {diam_out.stat().st_size/1024:.0f} KB')
    if kept == 0:
        raise SystemExit('no rows fell in the region. Check that the file is GRCh37.')

RAW_SUMSTATS['T2D'] = (diam_out, DIAMANTE_COLS, DIAMANTE_N)

print('\nready:', sorted(RAW_SUMSTATS))
missing_raw = [t for t in ['T2D', *CATALOG] if t not in RAW_SUMSTATS]
assert not missing_raw, f'still missing: {missing_raw}'

# DIAMANTE writes alleles in lower case and the Catalog files in upper case. orient() upper
# cases both sides, so nothing needs converting here.
_probe = pd.read_csv(diam_out, sep='\t', nrows=3)
print('\nfirst rows of the slice:')
print(_probe[[DIAMANTE_COLS[k] for k in ('id_col', 'ea', 'oa', 'eaf', 'beta', 'se')]]
      .to_string(index=False))


---
## 4. Fetch the 1000 Genomes region

`tabix` reads the remote index and pulls only the bytes for the interval, so this is a couple
of megabytes rather than the 1.2 GB chromosome.

```bash
VCF=https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr5.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz
tabix -h "$VCF" 5:55736375-55976375 | bgzip -c > region.vcf.gz
wget https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/integrated_call_samples_v3.20130502.ALL.panel
```


In [ ]:
# @title Stream the region and build the EUR sample list
BASE  = 'https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/'
VCF   = BASE + f'ALL.chr{CHROM}.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz'
PANEL = BASE + 'integrated_call_samples_v3.20130502.ALL.panel'

region_vcf = WORK / 'region.vcf.gz'
panel_file = WORK / 'samples.panel'

if not panel_file.exists():
    print('fetching the sample panel')
    urllib.request.urlretrieve(PANEL, panel_file)

# You already have the whole chromosome, so prefer it over streaming.
local_vcf = next((p for p in list(HERE.glob(f'ALL.chr{CHROM}.phase3*.vcf.gz'))
                  + list(ROOT.glob(f'ALL.chr{CHROM}.phase3*.vcf.gz'))
                  + list(DAT.glob(f'ALL.chr{CHROM}.phase3*.vcf.gz'))), None)

if not region_vcf.exists() or region_vcf.stat().st_size < 10_000:
    if local_vcf:
        print(f'using the local VCF: {local_vcf.name} '
              f'({local_vcf.stat().st_size/1e9:.2f} GB)')
        tbi = Path(str(local_vcf) + '.tbi')
        if not tbi.exists():
            # The index is a few hundred KB, far quicker to fetch than to rebuild.
            try:
                print('no .tbi alongside it, downloading the index')
                urllib.request.urlretrieve(VCF + '.tbi', tbi)
            except Exception as exc:
                print(f'  download failed ({exc}), building the index locally instead, '
                      'this takes a few minutes')
                run(f'{TABIX} -p vcf "{local_vcf}"', check=False)
        source = f'"{local_vcf}"'
    else:
        print('no local VCF found, streaming the region from the 1000 Genomes FTP')
        source = f'"{VCF}"'

    rc = run(f'{TABIX} -h {source} {CHROM}:{START}-{END} | bgzip -c > {region_vcf}',
             check=False).returncode
    if rc != 0 or not region_vcf.exists() or region_vcf.stat().st_size < 10_000:
        raise SystemExit(
            'tabix could not slice the VCF. If you were streaming, this is usually an htslib\n'
            'built without libcurl. Options:\n'
            f'  a) bcftools view -r {CHROM}:{START}-{END} {source} -Oz -o {region_vcf}\n'
            f'  b) index the local file first: tabix -p vcf ALL.chr{CHROM}.*.vcf.gz\n'
            f'  c) download the index: wget "{VCF}.tbi"')
else:
    print(f'reusing {region_vcf.name}')

panel = pd.read_csv(panel_file, sep=r'\s+')
assert 'super_pop' in panel.columns and 'sample' in panel.columns, \
    f'unexpected panel columns: {list(panel.columns)}. Delete {panel_file} and rerun.'
eur_panel = set(panel.loc[panel['super_pop'] == SUPERPOP, 'sample'])
assert eur_panel, f'no {SUPERPOP} samples in the panel'

# Read the sample ids straight out of the VCF header. The keep file is written in section 5,
# from the .fam PLINK actually produces, so its FID and IID cannot drift from PLINK's own
# convention. Guessing that convention is what produced the earlier "all samples excluded".
import gzip as _gzip
with _gzip.open(region_vcf, 'rt') as fh:
    chrom_line = next((l for l in fh if l.startswith('#CHROM')), None)
assert chrom_line, (f'{region_vcf} has no #CHROM header line, so it carries no samples. '
                    'Delete it and rerun this cell, making sure tabix is given -h.')
vcf_samples = chrom_line.rstrip('\n').split('\t')[9:]
eur_samples = [s for s in vcf_samples if s in eur_panel]

print(f'region VCF {region_vcf.stat().st_size/1e6:.2f} MB, '
      f'{len(vcf_samples)} samples in it')
print(f'{len(eur_panel)} {SUPERPOP} samples in the panel, {len(eur_samples)} of them present')
assert eur_samples, ('no overlap between the VCF samples and the panel. First VCF ids: '
                     f'{vcf_samples[:3]}, first panel ids: {sorted(eur_panel)[:3]}')


---
## 5. PLINK: genotypes to a signed correlation matrix

`--keep-allele-order` stops PLINK reassigning A1 to the minor allele. A1 is the reference
for every sign downstream, so it has to stay put.

In [ ]:
# @title The variant list, then --r square
# Everything downstream is keyed on chr:pos in GRCh37, never on rsID. The 1000G VCF leaves
# many IDs as '.', and the GWAS Catalog files do not all put rsIDs in variant_id, so an
# ID-based join silently loses variants. Position is the one key every source agrees on.

order0 = list(pd.read_csv(VARIANT_LIST, sep='\t').SNP)
assert len(order0) == len(set(order0)), 'variant_list.txt has duplicate rsIDs'
print(f'{len(order0)} target variants from {VARIANT_LIST.name}\n')

# ---- pass 1: import every sample and variant -----------------------------------
run(f'{PLINK} --vcf {region_vcf} --double-id --snps-only just-acgt '
    f'--biallelic-only strict --keep-allele-order --allow-no-sex '
    f'--make-bed --out {WORK}/region_all', quiet=True)

BIM_COLS = ['CHR', 'SNP', 'CM', 'BP', 'A1', 'A2']
bim_all = pd.read_csv(WORK / 'region_all.bim', sep=r'\s+', header=None,
                      names=BIM_COLS, dtype={'CHR': str, 'SNP': str, 'BP': int})
print(f'pass 1: {len(bim_all)} variants imported')

# rsIDs the VCF does carry, kept only to widen the rsID -> position lookup below
vcf_rsid_to_key = {r.SNP: f'{r.CHR}:{r.BP}'
                   for r in bim_all.itertuples() if str(r.SNP).startswith('rs')}
print(f'        {len(vcf_rsid_to_key)} of them carry an rsID, the rest are "."')

# ---- rewrite the .bim so every variant has a unique positional id --------------
bim_all['SNP'] = bim_all['CHR'] + ':' + bim_all['BP'].astype(str)
bim_all[BIM_COLS].to_csv(WORK / 'region_all.bim', sep='\t', header=False, index=False)

# Re-read it. Silently failing to rewrite the .bim is what made --extract match nothing
# useful last time, so this is checked rather than assumed.
_check = pd.read_csv(WORK / 'region_all.bim', sep=r'\s+', header=None, names=BIM_COLS,
                     dtype={'CHR': str, 'SNP': str, 'BP': int})
assert (_check.SNP == _check.CHR + ':' + _check.BP.astype(str)).all(), \
    'the .bim rewrite did not persist. Check write permissions on ' + str(WORK)
n_dup_pos = int(_check.SNP.duplicated().sum())
print(f'        .bim ids rewritten to chr:pos, {n_dup_pos} duplicated position(s)')

# ---- rsID -> chr:pos, from DIAMANTE first and the VCF second -------------------
diam_path, diam_cols, _ = RAW_SUMSTATS['T2D']
diam = pd.read_csv(diam_path, sep='\t',
                   usecols=[diam_cols['id_col'], diam_cols['chrom'], diam_cols['pos']],
                   dtype={diam_cols['chrom']: str, diam_cols['pos']: int})
diam[diam_cols['id_col']] = diam[diam_cols['id_col']].astype(str).str.strip()
rsid_to_key = dict(zip(diam[diam_cols['id_col']],
                       diam[diam_cols['chrom']] + ':' + diam[diam_cols['pos']].astype(str)))
from_diamante = sum(1 for s in order0 if s in rsid_to_key)
for rs, key in vcf_rsid_to_key.items():          # fill gaps, do not override DIAMANTE
    rsid_to_key.setdefault(rs, key)
resolved = {s: rsid_to_key[s] for s in order0 if s in rsid_to_key}
print(f'\nrsID -> position: {from_diamante} from DIAMANTE, '
      f'{len(resolved) - from_diamante} more from the VCF, {len(resolved)}/{len(order0)} total')
unresolved = [s for s in order0 if s not in resolved]
if unresolved:
    print(f'  no position for {len(unresolved)}: {unresolved[:8]}')

# keys that actually exist in the genotypes
present_keys = set(_check.SNP)
target_keys = sorted({k for k in resolved.values() if k in present_keys},
                     key=lambda k: int(k.split(':')[1]))
absent = [s for s, k in resolved.items() if k not in present_keys]
print(f'  {len(target_keys)} of them are in the 1000G region, {len(absent)} are not'
      + (f': {absent[:6]}' if absent else ''))
assert target_keys, ('no target variant matched a 1000G position. If the trait files are '
                     'GRCh38 and the VCF is GRCh37 this is exactly what it looks like.')

key_to_rsid = {k: s for s, k in resolved.items() if k in present_keys}
(WORK / 'snps.txt').write_text('\n'.join(target_keys) + '\n')

# ---- pass 2: restrict to EUR samples and those positions -----------------------
fam = pd.read_csv(WORK / 'region_all.fam', sep=r'\s+', header=None,
                  names=['FID', 'IID', 'PAT', 'MAT', 'SEX', 'PHENO'], dtype=str)
keep_rows = fam[fam.IID.str.strip().isin(eur_samples) | fam.FID.str.strip().isin(eur_samples)]
assert not keep_rows.empty, (f'no EUR sample matched the .fam. ids look like '
                             f'{list(fam.IID[:3])}, wanted {eur_samples[:3]}')
keep_rows[['FID', 'IID']].to_csv(WORK / 'eur.keep', sep='\t', header=False, index=False)
print(f'\nkeeping {len(keep_rows)} {SUPERPOP} samples')

run(f'{PLINK} --bfile {WORK}/region_all --keep {WORK}/eur.keep '
    f'--extract {WORK}/snps.txt --keep-allele-order --allow-no-sex '
    f'--make-bed --out {WORK}/region_eur', quiet=True)

n_kept = sum(1 for _ in open(WORK / 'region_eur.bim'))
print(f'after --extract: {n_kept} variants (asked for {len(target_keys)})')
assert n_kept != len(bim_all), (
    f'--extract kept all {n_kept} variants, so it matched nothing specific. The .bim ids and '
    'snps.txt are out of step.')
assert n_kept <= len(target_keys) + n_dup_pos, (
    f'--extract returned {n_kept}, more than the {len(target_keys)} requested.')
if n_kept < 0.8 * len(target_keys):
    print(f'  NOTE: {len(target_keys) - n_kept} requested positions were dropped by PLINK, '
          'usually multi-allelic or non-SNP records filtered in pass 1.')

# ---- duplicates, then the signed matrix ---------------------------------------
dupfile = WORK / 'region_eur.dupvar'
dupfile.unlink(missing_ok=True)
run(f'{PLINK} --bfile {WORK}/region_eur --list-duplicate-vars suppress-first '
    f'--out {WORK}/region_eur', check=False, quiet=True)

exclude, dropped_dups = '', set()
if dupfile.exists():
    lines = [l.strip() for l in dupfile.read_text().splitlines() if l.strip()]
    if lines and lines[0].split()[:1] == ['CHR']:      # header form, older PLINK
        lines = [l.split()[-1] for l in lines[1:]]
    dropped_dups = set(lines)
    if dropped_dups:
        exclude = f'--exclude {dupfile} '
        print(f'excluding {len(dropped_dups)} duplicated position(s)')

run(f'{PLINK} --bfile {WORK}/region_eur {exclude}--keep-allele-order --allow-no-sex '
    f'--r square spaces --out {WORK}/signed', quiet=True)
run(f'{PLINK} --bfile {WORK}/region_eur {exclude}--keep-allele-order --allow-no-sex '
    f'--freq --out {WORK}/region_eur', quiet=True)
print('signed matrix written')


In [ ]:
# @title Load it
R_plink = np.loadtxt(WORK / 'signed.ld')
R_plink = np.nan_to_num(R_plink, nan=0.0)
np.fill_diagonal(R_plink, 1.0)

bim = pd.read_csv(WORK / 'region_eur.bim', sep=r'\s+', header=None,
                  names=['chr', 'snp', 'cm', 'pos', 'A1', 'A2'],
                  dtype={'chr': str, 'snp': str, 'pos': int})
if dropped_dups:
    bim = bim[~bim.snp.isin(dropped_dups)].reset_index(drop=True)
assert len(bim) == R_plink.shape[0], (
    f'the .bim has {len(bim)} rows but the matrix is {R_plink.shape[0]} wide')

# keys stay positional. rsIDs are attached for labelling only.
keys = list(bim.snp)
assert all(':' in k for k in keys), f'.bim ids are not chr:pos, e.g. {keys[:3]}'
assert len(set(keys)) == len(keys), 'duplicate positions survived into the matrix'

rsid_of = {k: key_to_rsid.get(k, k) for k in keys}
pos_of_key = dict(zip(bim.snp, bim.pos))
ref_allele = dict(zip(bim.snp, bim.A1))
alt_allele = dict(zip(bim.snp, bim.A2))

frq = pd.read_csv(WORK / 'region_eur.frq', sep=r'\s+')
frq = frq[~frq.SNP.isin(dropped_dups)] if dropped_dups else frq
a1_freq = dict(zip(frq.SNP, frq.MAF))

print(f'signed matrix: {R_plink.shape[0]} x {R_plink.shape[1]}')
print(f'  range [{R_plink.min():.3f}, {R_plink.max():.3f}], '
      f'negative entries {int((R_plink < 0).sum()):,}')
print(f'  min eigenvalue {np.linalg.eigvalsh(R_plink).min():.4f}')
print(f'  {sum(1 for k in keys if rsid_of[k] != k)} of {len(keys)} carry a known rsID')
missing = [s for s in order0 if s not in set(rsid_of.values())]
print(f'  target variants not in the matrix: {len(missing)}'
      + (f' -> {missing[:8]}' if missing else ''))


---
## 6. Allele harmonisation

Every beta is flipped onto PLINK's A1. Both alleles are checked, so a genuine allele
mismatch is caught rather than silently strand-flipped. Strand-ambiguous variants, A/T and
C/G, are settled on frequency and dropped when too close to one half to call.

In [ ]:
# @title The rule
COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

def orient(effect, other, ref, alt, eaf=None, ref_freq=None, margin=AMBIGUOUS_MAF_MARGIN):
    """Sign to put BETA on `ref`. Returns (sign, note) or (None, reason)."""
    e, o = str(effect).upper(), str(other).upper()
    if e not in COMPLEMENT or o not in COMPLEMENT or e == o:
        return None, 'not a simple biallelic SNP'
    if {e, o} == {ref, alt}:
        if {e, o} in ({'A', 'T'}, {'C', 'G'}):
            if eaf is None or ref_freq is None:
                return None, 'strand ambiguous, no frequencies to resolve it'
            if abs(eaf - 0.5) < margin or abs(ref_freq - 0.5) < margin:
                return None, 'strand ambiguous, frequency too close to 0.5'
            return (1, 'ambiguous, resolved on frequency') \
                if (eaf - 0.5) * (ref_freq - 0.5) > 0 else \
                   (-1, 'ambiguous, resolved on frequency')
        return (1, 'direct') if e == ref else (-1, 'direct, flipped')
    ec, oc = COMPLEMENT[e], COMPLEMENT[o]
    if {ec, oc} == {ref, alt}:
        return (1, 'strand flip') if ec == ref else (-1, 'strand flip, flipped')
    return None, f'alleles do not match ({e}/{o} against {ref}/{alt})'


assert orient('A', 'G', 'A', 'G')[0] == 1
assert orient('G', 'A', 'A', 'G')[0] == -1
assert orient('T', 'C', 'A', 'G')[0] == 1
assert orient('C', 'T', 'A', 'G')[0] == -1
assert orient('a', 'g', 'A', 'G')[0] == 1          # DIAMANTE writes alleles in lower case
assert orient('A', 'C', 'A', 'G')[0] is None
assert orient('C', 'G', 'A', 'T')[0] is None
assert orient('A', 'T', 'A', 'T')[0] is None
assert orient('A', 'T', 'A', 'T', eaf=0.20, ref_freq=0.25)[0] == 1
assert orient('A', 'T', 'A', 'T', eaf=0.20, ref_freq=0.75)[0] == -1
assert orient('A', 'T', 'A', 'T', eaf=0.48, ref_freq=0.52)[0] is None
print('harmonisation self test passed')


In [ ]:
# @title Apply to the four traits
def build_key(df, cols):
    """chr:pos in the file's own coordinates."""
    return (df[cols['chrom']].astype(str).str.replace('^chr', '', regex=True).str.strip()
            + ':' + df[cols['pos']].astype(float).astype(int).astype(str))


def build_report(df, cols, keys_wanted):
    """If a file does not overlap, say whether it looks like a different assembly."""
    theirs = set(df['_key'])
    common = theirs & set(keys_wanted)
    if common:
        return f'{len(common)} positions overlap'
    ours = {int(k.split(':')[1]) for k in keys_wanted}
    mine = sorted(int(k.split(':')[1]) for k in theirs if k.split(':')[0] == str(CHROM))
    if not mine:
        chroms = sorted({k.split(':')[0] for k in theirs})[:5]
        return f'no rows on chromosome {CHROM}. Chromosomes present: {chroms}'
    # the hg19 to hg38 shift is constant within a locus, so a single modal offset shows it
    offsets = {}
    for p in mine[:400]:
        for q in ours:
            d = q - p
            if abs(d) < 5_000_000:
                offsets[d] = offsets.get(d, 0) + 1
    if offsets:
        best, hits = max(offsets.items(), key=lambda kv: kv[1])
        if hits >= 10:
            return (f'0 overlap, but {hits} positions line up at a constant offset of '
                    f'{best:+,} bp. This file is on a different genome build.')
    return '0 overlap and no constant offset. Check the chromosome and build.'


harmonised, dropped = {}, {}
for name, (path, cols, N) in RAW_SUMSTATS.items():
    raw = pd.read_csv(path, sep='\t')
    raw['_key'] = build_key(raw, cols)
    raw = raw.drop_duplicates(subset='_key').set_index('_key')

    print(f'{name:17s} {len(raw):5d} rows, {build_report(raw.reset_index(), cols, keys)}')

    eaf_col = cols['eaf'] if cols['eaf'] in raw.columns else None
    rows, notes = [], []
    for k in keys:
        if k not in raw.index:
            notes.append((rsid_of[k], 'no row at this position')); continue
        rec = raw.loc[k]
        eaf = float(rec[eaf_col]) if eaf_col and pd.notna(rec.get(eaf_col)) else None
        sign, note = orient(rec[cols['ea']], rec[cols['oa']], ref_allele[k], alt_allele[k],
                            eaf=eaf, ref_freq=a1_freq.get(k))
        if sign is None:
            notes.append((rsid_of[k], note)); continue
        rows.append({'KEY': k, 'SNP': rsid_of[k], 'BETA': sign * float(rec[cols['beta']]),
                     'SE': float(rec[cols['se']]), 'N': N, 'flipped': sign < 0})
    harmonised[name] = pd.DataFrame(rows, columns=['KEY', 'SNP', 'BETA', 'SE', 'N', 'flipped'])
    dropped[name] = notes
    print(f'{"":17s} kept {len(rows):4d}, flipped {sum(r["flipped"] for r in rows):4d}, '
          f'dropped {len(notes):4d}')

print('\n--- why variants were dropped ---')
for name, notes in dropped.items():
    if not notes:
        continue
    reasons = {}
    for _, why in notes:
        reasons[why.split('(')[0].strip()] = reasons.get(why.split('(')[0].strip(), 0) + 1
    print(f'{name}: ' + ', '.join(f'{v} {k}' for k, v in
                                  sorted(reasons.items(), key=lambda kv: -kv[1])[:4]))

common = set(keys)
for name, df in harmonised.items():
    if df.empty:
        print(f'\n{name} matched nothing. See its line above for whether this is a build '
              'mismatch or an allele problem.')
    common &= set(df['KEY'])

final = [k for k in keys if k in common]
if not final:
    raise SystemExit('0 variants shared across the matrix and all four traits.')

pos = {k: i for i, k in enumerate(keys)}
R_final = R_plink[np.ix_([pos[k] for k in final], [pos[k] for k in final])]
harmonised = {k: v.set_index('KEY').loc[final].reset_index() for k, v in harmonised.items()}
final_rsids = [rsid_of[k] for k in final]
print(f'\nshared across the matrix and all four traits: {len(final)} variants')


---
## 7. The diagnostic that tests the whole thing

Under a single causal variant, the Z of a correlated variant tracks `r * Z_lead`. Pairs at
high `|r|` whose Z scores contradict the sign of `r` are therefore a direct measure of
disagreement between the matrix and the summary statistics.

This is the check that a correctly signed matrix with unharmonised betas still fails, so it
tests the harmonisation and the matrix together. It is reported at three thresholds, before
and after, with a scatter of observed Z against `r * Z_lead`.


In [ ]:
# @title Before and after
import matplotlib.pyplot as plt

def concordance(Rm, Z, label):
    iu = np.triu_indices(len(Z), 1)
    r, zi, zj = Rm[iu], Z[iu[0]], Z[iu[1]]
    print('\n' + label)
    for thr in (0.9, 0.8, 0.6):
        m = np.abs(r) >= thr
        if not m.any():
            continue
        bad = np.sign(zi[m] * zj[m]) != np.sign(r[m])
        print(f'  |r| >= {thr}: {int(m.sum()):5d} pairs, contradicting the LD sign: '
              f'{int(bad.sum()):5d} ({bad.mean():.1%})')

# Reconstruct Z_before directly from our harmonised table using the 'flipped' column
df_t2d = harmonised['T2D']
Z_after  = (df_t2d['BETA'] / df_t2d['SE']).to_numpy()
# If it was flipped, multiply by -1 to revert to the original unharmonised Z
Z_before = Z_after * np.where(df_t2d['flipped'], -1, 1)

concordance(np.abs(R_final), Z_before, 'magnitudes corrected to |r|, sign and betas untouched')
concordance(R_final,         Z_before, 'signed r, but betas still unharmonised')
concordance(R_final,         Z_after,  'AFTER: signed r and harmonised betas')

lead = int(np.argmax(np.abs(Z_after)))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (Z, title) in zip(axes, [(Z_before, 'Before harmonisation'),
                                 (Z_after,  'After harmonisation')]):
    ax.scatter(R_final[lead] * Z[lead], Z, s=18, facecolor='none', edgecolor='dimgrey')
    lim = np.array([min(Z.min(), -1) * 1.05, max(Z.max(), 1) * 1.05])
    ax.plot(lim, lim, ls='dashed', lw=0.9, color='tab:red', label='Z = r x Z(lead)')
    
    lead_snp_id = df_t2d['SNP'].iloc[lead]
    ax.set_xlabel(f'r with {lead_snp_id}  x  its Z'); ax.set_ylabel('observed Z')
    ax.set_title(title, fontsize=10); ax.legend(fontsize=8)
    
fig.suptitle('LD mismatch diagnostic, T2D', y=1.02); fig.tight_layout()
fig.savefig(RESULTS / 'ld_mismatch_before_after.png', dpi=200, bbox_inches='tight')
plt.show()

---
## 8. Write the inputs

In [ ]:
# @title Outputs, all in one variant order
# Written into dat/ alongside the originals, with a _signed suffix so nothing is overwritten.
np.savetxt(DAT / 'rs3843467_signed.ld', R_final, fmt='%.6f', delimiter=' ')
for name, df in harmonised.items():
    df[['SNP', 'BETA', 'SE', 'N']].to_csv(DAT / f'{name}_bse_signed.txt',
                                          sep='\t', header=True, index=False)

pd.DataFrame({
    'row': range(len(final)),
    'SNP': final_rsids,
    'chromosome': CHROM,
    'position': [pos_of_key[k] for k in final],
    'reference_allele': [ref_allele[k] for k in final],   # PLINK A1, what every sign refers to
    'other_allele': [alt_allele[k] for k in final],
    'A1_freq_EUR': [a1_freq.get(k, np.nan) for k in final],
    'T2D_beta_flipped': harmonised['T2D'].flipped.to_numpy(),
}).to_csv(DAT / 'variant_order.tsv', sep='\t', index=False)

# CARMA also accepts a .z style file
for name, df in harmonised.items():
    z = pd.DataFrame({
        'rsid': df.SNP, 'chromosome': CHROM,
        'position': [pos_of_key[k] for k in df.KEY],
        'allele1': [ref_allele[k] for k in df.KEY],
        'allele2': [alt_allele[k] for k in df.KEY],
        'maf': [a1_freq.get(k, np.nan) for k in df.KEY],
        'beta': df.BETA, 'se': df.SE})
    z.to_csv(DAT / f'{name}_signed.z', sep='\t', header=True, index=False)

print('written to', DAT.resolve())
for f in sorted(DAT.glob('*_signed*')) + [DAT / 'variant_order.tsv']:
    print(f'   {f.name:34s} {f.stat().st_size/1024:8.1f} KB')
print('\nvariant_order.tsv is the row index of the matrix and records the allele every sign '
      'and every beta refers to. The matrix carries no labels of its own, so deposit the two '
      'together.')


---
## 9. Clean up

Deletes everything that was only needed to get here. What survives is the four files the
downstream tools read, the diagnostic plot, and the small region slices that let you rerun the
harmonisation without downloading anything again.

The 1000 Genomes chromosome 5 VCF is left alone by default, since it is 1.2 GB and you may want
it for something else. Set `DELETE_CHR5_VCF = True` to remove it too.


In [ ]:
# @title Remove the intermediates
DELETE_CHR5_VCF = False        # the 1.2 GB ALL.chr5.*.vcf.gz you already had
DRY_RUN         = True         # set to False to actually delete

KEEP = {
    DAT / 'rs3843467_signed.ld',
    DAT / 'variant_order.tsv',
    *(DAT / f'{n}_bse_signed.txt' for n in RAW_SUMSTATS),
    *(DAT / f'{n}_signed.z' for n in RAW_SUMSTATS),
    RESULTS / 'ld_mismatch_before_after.png',
    *(p for p in RAW_DIR.glob('*.tsv')),                 # the small region slices
    VARIANT_LIST,                                        # the input list, untouched
}

doomed = []
for pattern in ('region.vcf.gz', 'region_all.*', 'region_eur.*', 'signed.ld',
                'signed.log', 'signed.nosex',
                'snps.txt', 'samples.panel', 'eur.keep', 'plink.zip', '*.log', '*.nosex'):
    doomed += [p for p in WORK.glob(pattern) if p.is_file() and p not in KEEP]
doomed += [p for p in BIN.rglob('*') if p.is_file()]     # the downloaded plink binary

if DELETE_CHR5_VCF:
    for place in (HERE, ROOT, DAT):
        doomed += [p for p in place.glob(f'ALL.chr{CHROM}.phase3*.vcf.gz*')]

doomed = sorted({p for p in doomed if p.exists() and p not in KEEP})
total = sum(p.stat().st_size for p in doomed)

def show(p):
    """Path relative to the project root when possible, absolute otherwise."""
    try:
        return Path(p).resolve().relative_to(ROOT.resolve())
    except ValueError:
        return Path(p).resolve()

print(f'{"WOULD DELETE" if DRY_RUN else "DELETING"} {len(doomed)} files, '
      f'{total/1e6:.1f} MB\n')
for p in doomed:
    print(f'   {p.stat().st_size/1e6:8.2f} MB  {show(p)}')
    if not DRY_RUN:
        p.unlink()
if not DRY_RUN:
    for d in (BIN,):
        if d.exists() and not any(d.iterdir()):
            d.rmdir()

print('\nKEEPING:')
for p in sorted(KEEP):
    if Path(p).exists():
        print(f'   {Path(p).stat().st_size/1024:8.1f} KB  {show(p)}')

if DRY_RUN:
    print('\nThis was a dry run. Set DRY_RUN = False and rerun to delete.')


---
## 10. Run SharePro and CARMA

Run both from `2_colocalization_and_finemapping/`, the folder that contains `dat/`.

```bash
python src/SharePro/sharepro_coloc.py \
  --z dat/fasting_insulin_bse_signed.txt \
       dat/T2D_bse_signed.txt \
       dat/fasting_glucose_bse_signed.txt \
  --ld dat/rs3843467_signed.ld \
  --save 2a_colocalization/results/res_signed --K 10

Rscript 2b_finemapping/CARMA_T2D_signed.R
```

### Where everything lands

Each stage reads its inputs from `dat/`, keeps scratch in its own `interim/`, and writes
final output to its own `results/`. Section 8 of this notebook is the only step that writes
into `dat/`.

```
2_colocalization_and_finemapping/
  dat/                        inputs only, shared by both stages
    raw/                        region slices of the source sumstats
  2a_colocalization/
    interim/                    PLINK scratch, safe to delete
    results/                    res_signed.sharepro.txt, ld_mismatch_before_after.png
  2b_finemapping/
    interim/                    CARMA post_* dumps, safe to delete
    results/                    carma_results_*.csv, credible_set_variants_*.csv,
                                shared_high_pip_variants_signed.csv
```

### What to expect from the credible sets

rs256903 does not separate from rs459193, rs173964 and rs256904. They sit at pairwise R2 above
0.99, so no summary-statistic method can rank them on association evidence alone. Expect a
credible set spanning the 3' end haplotype, not a winner among its members. The prioritisation
of rs256903 rests on the sequence models, which never use an LD matrix.

The LD matrix is rank-deficient, 149 independent dimensions across 247 variants, with 347
pairs at `|r| = 1`. CARMA searches models by shotgun stochastic search and picks one
representative per collinear group, so `set.seed` in the R script is what makes a run
reproducible. Which member of a group it lands on is not evidence about that member.

### Two things to watch

`outlier.switch` is `TRUE` in the CARMA script. It is CARMA's own test for summary statistics
that disagree with the supplied LD. Variants it flags are informative about the inputs.

The colocalization share for the model including two hour glucose is worth recording alongside
the model without it, since the manuscript quotes that number.

### If a variant gets dropped that you need

The usual causes are a strand-ambiguous variant too close to 50% frequency, or a variant
absent from 1000G EUR. Both are listed by section 6. Widening `HALF_WIDTH` helps with neither.
For an ambiguous variant that matters, check its allele frequency against the GWAS by hand and
add it back rather than lowering `AMBIGUOUS_MAF_MARGIN` across the board.
